In [ ]:
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM
import torch

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
      model_name,
      torch_dtype=torch.float16
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [ ]:
def generate_questions(domain):

    prompt = f"""

    Generate exactly 4 interview questions.

    Domain = {domain}

    Rules:

    1 beginner question

    1 technical question

    1 coding question

    1 project question

    Return only questions
    """

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=300
    )

    result = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return result

In [ ]:
def evaluate_answer(question, answer):

    prompt = f"""

    You are an interviewer.

    Interview Question:

    {question}

    Candidate Answer:

    {answer}

    Evaluate answer on:

    Technical correctness

    Clarity

    Communication

    Confidence

    Give score out of 10

    Give suggestions

    """

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=500
    )

    feedback = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return feedback

In [ ]:
!pip install sentence-transformers streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 56.0 MB/s eta 0:00:00


In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

embed = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

def score_similarity(
    expected,
    user_answer
):

    e1 = embed.encode(expected)

    e2 = embed.encode(user_answer)

    score = cosine_similarity(
        [e1],
        [e2]
    )

    return score

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
def final_score(
    llm_score,
    similarity
):

    total = (
        llm_score * 0.7
        +
        similarity * 10 * 0.3
    )

    return total

In [ ]:
{
 "user":"student_1",
 "domain":"python",
 "question":"What is tuple",
 "answer":"Tuple is immutable",
 "score":8.4,
 "feedback":"Good answer"
}

{'user': 'student_1',
 'domain': 'python',
 'question': 'What is tuple',
 'answer': 'Tuple is immutable',
 'score': 8.4,
 'feedback': 'Good answer'}

In [ ]:
from fastapi import FastAPI

app = FastAPI()

@app.post("/generate")

def generate(data: dict):

    return {
      "questions":
      generate_questions(
          data["domain"]
      )
    }


@app.post("/evaluate")

def evaluate(data: dict):

    feedback = evaluate_answer(
        data["question"],
        data["answer"]
    )

    return {
      "feedback": feedback
    }

In [ ]:
import streamlit as st

st.title(
   "AI Interview Coach"
)

domain = st.text_input(
    "Enter domain"
)

if st.button("Generate"):

    # call API
    pass

2026-06-24 07:48:00.165 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-24 07:48:00.436 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-06-24 07:48:00.443 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-24 07:48:00.445 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-24 07:48:00.453 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-24 07:48:00.458 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-24 07:48:00.462 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-06-24 07:48:00.471 Thread 'MainThread': mi